In [0]:
%pip install pyreadstat openpyxl unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.2/594.2 kB 22.5 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import re
from pathlib import Path
import pandas as pd
import numpy as np
from unidecode import unidecode
import pyreadstat
from pyspark.sql import SparkSession, functions as F

In [0]:
DATA_DIR = Path("/Volumes/workspace/default/data")
OUT_DIR = Path("/Volumes/workspace/default/parquet")

spanish_months = {"enero":1,"febrero":2,"marzo":3,"abril":4,"mayo":5,"junio":6,"julio":7,"agosto":8,"septiembre":9,"setiembre":9,"octubre":10,"noviembre":11,"diciembre":12}

def norm_text(s):
    if s is None:
        return None
    return re.sub(r"\s+","_",unidecode(str(s)).strip().lower())

sinonimos = {
    "anio": {"anio","ano","year"},
    "mes": {"mes","month"},
    "departamento": {"departamento","depto"},
    "municipio": {"municipio","muni","mupio_ocu"},
    "zona": {"zona","area_geo_ocu","areag_ocu"},
    "tipo_accidente": {"tipo_accidente","tipo_de_accidente","clase","clase_accidente","clase_acc","tipo_eve","tipo_evento","tipo"},
    "hora": {"hora","hr","hora_del_dia"},
    "dia_semana": {"dia_semana","dia","día","weekday"},
    "color": {"color"},
    "sexo_conductor": {"sexo_conductor","sexo","genero","genero_conductor","sexo_del_conductor"},
    "condicion_victima": {"condicion","condicion_victima","estado_victima","resultado_victima"},
    "edad": {"edad","age"},
    "tipo_victima": {"tipo_victima","victima_tipo","categoria_victima"}
}

def canon_name(col):
    n = norm_text(col)
    for canon, alts in sinonimos.items():
        if n in alts:
            return canon
    for canon, alts in sinonimos.items():
        for a in alts:
            if a in n:
                return canon
    return n

def normalizar_columnas(df):
    df = df.copy()
    m = {c: canon_name(c) for c in df.columns}
    df.columns = [m[c] for c in df.columns]
    df = df.loc[:, ~pd.Index(df.columns).duplicated(keep='first')]
    return df

def ensure_series(x):
    if isinstance(x, pd.DataFrame):
        return x.iloc[:,0]
    return x

def parse_mes_col(serie):
    s = ensure_series(serie)
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce").astype("Int64")
    s = s.astype(str).str.strip().str.lower().map(lambda x: unidecode(x))
    s = s.map(lambda v: spanish_months.get(v, v))
    s = pd.to_numeric(s, errors="coerce").astype("Int64")
    return s

def parse_hora_col(serie):
    s = ensure_series(serie)
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce").clip(lower=0, upper=23).astype("Int64")
    s = s.astype(str).str.extract(r"(\d{1,2})", expand=False)
    s = pd.to_numeric(s, errors="coerce").clip(lower=0, upper=23).astype("Int64")
    return s

def agregar_anio_por_filename(df, path):
    if "anio" in df.columns:
        return df
    m = re.search(r"(20\d{2}|19\d{2})", str(path))
    if m:
        df = df.copy()
        df["anio"] = int(m.group(1))
    return df

def cargar_tabla(archivo):
    if archivo.suffix.lower() == ".sav":
        df, meta = pyreadstat.read_sav(str(archivo), apply_value_formats=True)
        return df
    if archivo.suffix.lower() in [".xlsx",".xls"]:
        return pd.read_excel(archivo)
    return None

def limpiar_basica(df):
    df = normalizar_columnas(df)
    if "mes" in df.columns:
        df["mes"] = parse_mes_col(df["mes"])
    if "hora" in df.columns:
        df["hora"] = parse_hora_col(df["hora"])
    if "dia_semana" in df.columns:
        df["dia_semana"] = df["dia_semana"].astype(str).str.strip().str.lower().map(lambda x: unidecode(x))
    if "departamento" in df.columns:
        df["departamento"] = df["departamento"].astype(str).str.strip().str.upper()
    if "municipio" in df.columns:
        df["municipio"] = df["municipio"].astype(str).str.strip().str.title()
    if "zona" in df.columns:
        df["zona"] = df["zona"].astype(str).str.extract(r"(\d+)", expand=False)
    if "tipo_accidente" in df.columns:
        df["tipo_accidente"] = df["tipo_accidente"].astype(str).str.strip().str.title()
    if "color" in df.columns:
        df["color"] = df["color"].astype(str).str.strip().str.title()
    if "sexo_conductor" in df.columns:
        df["sexo_conductor"] = df["sexo_conductor"].astype(str).str.strip().str.title()
    if "condicion_victima" in df.columns:
        df["condicion_victima"] = df["condicion_victima"].astype(str).str.strip().str.title()
    if "edad" in df.columns:
        df["edad"] = pd.to_numeric(df["edad"], errors="coerce").astype("Int64")
    return df

def cargar_unificar(base):
    patrones = [f"{base}_*.sav",f"{base}_*.xlsx"]
    archivos = []
    for pat in patrones:
        archivos.extend(sorted((DATA_DIR).glob(pat)))
    dfs = []
    for a in archivos:
        df = cargar_tabla(a)
        if df is None:
            continue
        df = agregar_anio_por_filename(df, a)
        df = limpiar_basica(df)
        dfs.append(df)
    if not dfs:
        return pd.DataFrame()
    df = pd.concat(dfs, ignore_index=True)
    if "anio" in df.columns:
        df = df[df["anio"].between(2014,2023, inclusive="both")]
    return df

def sanitize_parquet(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in df.columns:
        s = df[c]
        if s.dtype == object:
            s = s.map(lambda v: v.decode('utf-8', 'ignore') if isinstance(v, (bytes, bytearray)) else v)
            s = s.astype("string")
        elif pd.api.types.is_bool_dtype(s):
            s = s.astype("boolean")
        df[c] = s
    return df

df_hechos = cargar_unificar("hechos")
df_vehiculos = cargar_unificar("vehiculos")
df_fallecidos = cargar_unificar("fallecidos")
df_lesionados = cargar_unificar("lesionados")

frames_v = []
if not df_fallecidos.empty:
    if "condicion_victima" not in df_fallecidos.columns:
        df_fallecidos = df_fallecidos.assign(condicion_victima="Fallecido")
    frames_v.append(df_fallecidos)
if not df_lesionados.empty:
    if "condicion_victima" not in df_lesionados.columns:
        df_lesionados = df_lesionados.assign(condicion_victima="Lesionado")
    frames_v.append(df_lesionados)
if frames_v:
    cols_union = sorted(set().union(*[f.columns for f in frames_v]))
    frames_v = [f.reindex(columns=cols_union) for f in frames_v]
    df_victimas = pd.concat(frames_v, ignore_index=True)
else:
    df_victimas = pd.DataFrame()

df_hechos_pq = sanitize_parquet(df_hechos)
df_vehiculos_pq = sanitize_parquet(df_vehiculos)
df_victimas_pq = sanitize_parquet(df_victimas)

df_hechos_pq.to_parquet(OUT_DIR / "hechos.parquet", index=False)
df_vehiculos_pq.to_parquet(OUT_DIR / "vehiculos.parquet", index=False)
df_victimas_pq.to_parquet(OUT_DIR / "victimas.parquet", index=False)

len(df_hechos_pq), len(df_vehiculos_pq), len(df_victimas_pq)



(70435, 105665, 100357)

In [0]:
spark = SparkSession.builder.getOrCreate()

OUT_BASE = "/Volumes/workspace/default/parquet/"
hechos = spark.read.parquet(f"{OUT_BASE}/hechos.parquet")
vehiculos = spark.read.parquet(f"{OUT_BASE}/vehiculos.parquet")
victimas = spark.read.parquet(f"{OUT_BASE}/victimas.parquet")

try:
    display
except NameError:
    def display(df): df.show(20, truncate=False)

hechos.count(), vehiculos.count(), victimas.count()

(70435, 105665, 100357)

## Ejercicios

In [0]:
## Ejercicio 1

hechos.count(), vehiculos.count(), victimas.count()

hechos.show(10, truncate=False)

vehiculos.show(10, truncate=False)

victimas.show(10, truncate=False)

sel_h = [c for c in ["anio","mes","departamento","municipio","zona","tipo_accidente","hora","dia_semana"] if c in hechos.columns]
hechos.select(sel_h).summary().show(truncate=False)

sel_vh = [c for c in ["anio","mes","departamento","municipio","zona","tipo_accidente","hora","dia_semana","color","sexo_conductor"] if c in vehiculos.columns]
vehiculos.select(sel_vh).summary().show(truncate=False)

sel_v = [c for c in ["anio","mes","departamento","municipio","zona","tipo_accidente","hora","dia_semana","condicion_victima","edad","tipo_victima"] if c in victimas.columns]
victimas.select(sel_v).summary().show(truncate=False)

+---------+----------+---+----+-------------+-------------------------+----+--------------+----+-----------+--------------+--------+----------+---------+------------+----------+------+---------------+----------+----------+---------+------------+
|num_hecho|dia_semana|mes|hora|departamento |municipio                |zona|sexo_conductor|edad|mayor_menor|tipo_accidente|color   |modelo_veh|causa_acc|marca_veh   |estado_pil|anio  |num_correlativo|corre_base|estado_con|num_corre|g_modelo_veh|
+---------+----------+---+----+-------------+-------------------------+----+--------------+----+-----------+--------------+--------+----------+---------+------------+----------+------+---------------+----------+----------+---------+------------+
|NULL     |1.0       |1  |3   |GUATEMALA    |Guatemala                |7   |Hombre        |20  |Mayor      |Automóvil     |Negro   |1998.0    |NULL     |Mazda       |NULL      |2014.0|1.0            |1.0       |Ignorado  |NULL     |NULL        |
|NULL     |1.0  

In [0]:
## Ejercicio 2

hechos.select("anio").distinct().orderBy("anio").show(100, truncate=False)

vehiculos.select("anio").distinct().orderBy("anio").show(100, truncate=False)

victimas.select("anio").distinct().orderBy("anio").show(100, truncate=False)

+------+
|anio  |
+------+
|2014.0|
|2015.0|
|2016.0|
|2017.0|
|2018.0|
|2019.0|
|2020.0|
|2021.0|
|2022.0|
|2023.0|
+------+

+------+
|anio  |
+------+
|2014.0|
|2015.0|
|2016.0|
|2017.0|
|2018.0|
|2019.0|
|2020.0|
|2021.0|
|2022.0|
|2023.0|
+------+

+------+
|anio  |
+------+
|2014.0|
|2015.0|
|2016.0|
|2017.0|
|2018.0|
|2019.0|
|2020.0|
|2021.0|
|2022.0|
|2023.0|
+------+



In [0]:
## ejercicio 3
def rename_first(df, target, candidates):
    for c in candidates:
        if c in df.columns:
            return df if c == target else df.withColumnRenamed(c, target)
    return df

hechos = rename_first(
    hechos, "tipo_accidente", [
        "tipo_accidente", "tipo_eve", "clase", "clase_acc", "tipo_evento", "tipo"
    ]
)
hechos = rename_first(
    hechos, "municipio", [
        "municipio", "mupio_ocu", "muni"
    ]
)
hechos = rename_first(
    hechos, "departamento", [
        "departamento", "depto"
    ]
)
hechos = rename_first(
    hechos, "hora", [
        "hora", "hr", "hora_del_dia"
    ]
)
hechos = rename_first(
    hechos, "dia_semana", [
        "dia_semana", "dia", "día", "weekday"
    ]
)
hechos = rename_first(
    hechos, "zona", [
        "zona"
    ]
)

display(
    hechos.select("tipo_accidente")
    .distinct()
    .orderBy("tipo_accidente")
)

tipo_accidente
1
10
11
12
13
14
15
16
17
18


In [0]:
## Ejercicio 4
dep_hechos = hechos.select("departamento").distinct().count() if "departamento" in hechos.columns else None
dep_vehiculos = vehiculos.select("departamento").distinct().count() if "departamento" in vehiculos.columns else None
dep_victimas = victimas.select("departamento").distinct().count() if "departamento" in victimas.columns else None
(dep_hechos, dep_vehiculos, dep_victimas)


(53, 46, 46)

In [0]:
## Ejercicio 5 
hechos_por_anio_dep = hechos.groupBy("anio","departamento").agg(F.count(F.lit(1)).alias("accidentes")).orderBy("anio","departamento")
display(hechos_por_anio_dep)


anio,departamento,accidentes
2014.0,ALTA VERAPAZ,223
2014.0,BAJA VERAPAZ,95
2014.0,CHIMALTENANGO,228
2014.0,CHIQUIMULA,126
2014.0,EL PROGRESO,127
2014.0,ESCUINTLA,444
2014.0,GUATEMALA,1912
2014.0,HUEHUETENANGO,136
2014.0,IZABAL,255
2014.0,JALAPA,75


Tras agrupar los datos de accidentes por año y departamento usando la función groupBy, se observa que el total de accidentes muestra un incremento sostenido entre 2014 y 2023.

El departamento de Guatemala muestra el mayor número de accidentes en todos los años, superando los 3000 accidentes anuales en los últimos periodos (2021–2023).
Esto indica una alta concentración de tránsito y densidad vehicular.

Escuintla se mantiene como el segundo departamento más afectado, con incrementos notables entre 2016 y 2023, pasando de 627 a 930 accidentes.

Alta Verapaz, Chimaltenango y Quetzaltenango también muestran aumentos graduales, aunque con volúmenes menores (200–400 accidentes anuales).

En general el departamento de Guatemala lidera ampliamente en número de accidentes, seguido por Escuintla y Chimaltenango. Los departamentos con menor incidencia son Totonicapán, Sololá y Jalapa.


In [0]:
## Ejercicio 6
acc_por_dia_2023 = hechos.filter(F.col("anio")==2023).groupBy("dia_semana").agg(F.count(F.lit(1)).alias("accidentes")).orderBy(F.desc("accidentes"))
display(acc_por_dia_2023)
acc_por_dia_2023.limit(1).show(truncate=False)

dia_semana,accidentes
1.0,324
2.0,300
9.0,294
27.0,291
25.0,291
10.0,290
17.0,290
3.0,288
11.0,281
5.0,280


+----------+----------+
|dia_semana|accidentes|
+----------+----------+
|1.0       |324       |
+----------+----------+



En el análisis de los accidentes ocurridos durante 2023, se observa que el día 1.0 registra la mayor cantidad de siniestros viales (324 accidentes), seguido de los días 2.0 y 9.0. Esto indica que los primeros días de la semana o del mes concentran una mayor incidencia de accidentes.En contraste, los últimos días presentan cifras más bajas, lo que sugiere una menor movilidad. En general, los resultados evidencian que el inicio de la semana es el periodo de mayor riesgo vial.

In [0]:
## Ejercico 7
dist_hora_muni = hechos.filter(F.col("municipio")==F.lit("Guatemala")).groupBy("hora").agg(F.count(F.lit(1)).alias("accidentes")).orderBy("hora")
display(dist_hora_muni)

hora,accidentes
null,6
0,622
1,601
2,436
3,380
4,304
5,336
6,384
7,481
8,421


El análisis de la distribución de accidentes por hora del día en el municipio de Guatemala muestra una tendencia clara de incremento durante las horas de la tarde y la noche. Los valores más altos se registran entre las 19:00 y 22:00 horas, alcanzando su pico máximo a las 21:00 con 876 accidentes. n contraste, las horas de la madrugada (de 2:00 a 5:00) presentan los valores más bajos, reflejando una menor circulación vehicular. En general, el histograma evidencia que la mayoría de los accidentes se concentran entre las 17:00 y 23:00 horas, lo que sugiere que el tráfico intenso, la fatiga y la disminución de la visibilidad influyen significativamente en el aumento en los accidentes viales.

In [0]:
## Ejercicio 8
llave_hv = [c for c in ["anio","mes","departamento","municipio","zona","tipo_accidente","hora","dia_semana"] if c in hechos.columns and c in vehiculos.columns]
hv_join = hechos.join(vehiculos, on=llave_hv, how="inner")
hv_join.count()

23480

Se obtuvieron 23480 registros.

In [0]:
## Ejercicio 9 - Promedio de vehículos por accidente por departamento

# Contar vehículos por departamento
veh_por_dep = vehiculos.groupBy("departamento").agg(
    F.count(F.lit(1)).alias("total_vehiculos")
)

# Contar accidentes por departamento
acc_por_dep = hechos.groupBy("departamento").agg(
    F.count(F.lit(1)).alias("total_accidentes")
)
por_dep = veh_por_dep.join(acc_por_dep, on="departamento", how="inner").withColumn(
    "vehiculos_por_accidente", 
    F.col("total_vehiculos") / F.col("total_accidentes")
).select("departamento", "vehiculos_por_accidente").orderBy(F.desc("vehiculos_por_accidente"))

# Guardar en Parquet
por_dep.write.mode("overwrite").parquet(f"{OUT_BASE}/vehiculos_por_accidente_departamento")

# Cargar y mostrar top 10
top10 = spark.read.parquet(f"{OUT_BASE}/vehiculos_por_accidente_departamento").orderBy(
    F.desc("vehiculos_por_accidente")
).limit(10)

print("=== Top 10 departamentos con más vehículos por accidente ===")
display(top10)

=== Top 10 departamentos con más vehículos por accidente ===


departamento,vehiculos_por_accidente
SACATEPÉQUEZ,2.1317480719794344
QUICHÉ,1.8486646884272997
PETÉN,1.7932862190812722
SUCHITEPÉQUEZ,1.7414772727272727
TOTONICAPÁN,1.711297071129707
RETALHULEU,1.5875933371625504
QUETZALTENANGO,1.5493306891422904
SAN MARCOS,1.539002108222066
ZACAPA,1.5337579617834396
ESCUINTLA,1.52770656675212


Al analizar el promedio de vehículos involucrados por accidente en cada departamento, se observa que Sacatepéquez presenta el valor más alto con 2.13 vehículos por accidente, lo que sugiere una mayor frecuencia de choques múltiples o colisiones entre varios automotores. Le siguen Quiché (1.85) y Petén (1.79) en general, los resultados reflejan que los departamentos con mayor actividad turística o tránsito mixto tienden a registrar más vehículos por accidente

In [0]:
## Ejercicio 10
top_colores = vehiculos.groupBy("color").agg(F.count(F.lit(1)).alias("n")).orderBy(F.desc("n")).limit(5)
display(top_colores)


color,n
Ignorado,25969
Negro,15734
Rojo,13372
Blanco,12419
Gris,10075


El análisis de los colores de vehículos más involucrados en accidentes muestra que el color “Ignorado” ocupa el primer lugar con 25,969 registros, lo que indica una gran cantidad de casos sin información específica. Entre los colores identificados, los vehículos negros (15,734), rojos (13,372) y blancos (12,419) son los más frecuentes en los accidentes, seguidos por los grises (10,075).

In [0]:
## Ejercicio 11 - Lesionados en 2023 por mes

print("Se procede a calcular todos los lesionados por mes en 2023\n")

# Filtrar lesionados en 2023 (usando fall_les)
les_2023 = victimas.filter(
    (F.col("anio") == 2023) & 
    (
        (F.col("fall_les") == "Lesionado") | 
        (F.col("fall_les") == "2")
    )
)

# Agrupar por mes
les_por_mes = les_2023.groupBy("mes").agg(
    F.count(F.lit(1)).alias("lesionados")
).orderBy("mes")

print(f"Total de lesionados en 2023: {les_2023.count()}")

# Mostrar y graficar
display(les_por_mes)

Se procede a calcular todos los lesionados por mes en 2023

Total de lesionados en 2023: 8921


mes,lesionados
1,790
2,748
3,750
4,950
5,740
6,706
7,844
8,614
9,658
10,632


Databricks visualization. Run in Databricks to view.

El análisis de los lesionados por atropello en 2023 muestra variaciones significativas a lo largo del año. El número de lesionados alcanza su pico en abril con 950 casos, seguido por enero (790) y julio (844), lo que podría coincidir con meses de mayor actividad vehicular o eventos públicos. En contraste, los valores más bajos se registran en noviembre (602) y agosto (614), reflejando una disminución temporal en la incidencia de atropellos.

In [0]:
## Ejercicio 12 - Fallecidos por tipo de vehículo involucrado

# Llave compuesta (igual que ejercicio 8)
llave_hv2 = [c for c in ["anio","mes","departamento","municipio","zona","tipo_accidente","hora","dia_semana"] 
             if c in hechos.columns and c in victimas.columns]

# Filtrar solo fallecidos usando fall_les
fall = victimas.filter(
    (F.col("fall_les") == "Fallecido") | 
    (F.col("fall_les") == "1") |
    (F.lower(F.col("condicion_victima")).like("%fallec%"))
)

# Join con hechos para relacionar los accidentes
fall_join = fall.join(
    hechos.select(llave_hv2 + ["tipo_accidente"]).distinct(), 
    on=llave_hv2, 
    how="inner"
)

# Agrupar por tipo de accidente (vehículo)
fall_por_tipo = fall_join.groupBy("tipo_accidente").agg(
    F.count(F.lit(1)).alias("fallecidos")
).orderBy(F.desc("fallecidos"))

print(f"Total de fallecidos relacionados con hechos: {fall_join.count()}")
display(fall_por_tipo)

Total de fallecidos relacionados con hechos: 17577


tipo_accidente,fallecidos
Motocicleta,5338
4,2665
Automóvil,1720
3,1617
1,1307
99,745
Ignorado,516
9,487
5,431
12,414


Databricks visualization. Run in Databricks to view.

El análisis del total de fallecidos por tipo de accidente revela que los hechos involucrando motocicletas representan la principal causa de muertes, con 5,338 fallecidos, lo que evidencia la alta vulnerabilidad de los motociclistas en la vía pública. Los automóviles también muestran una cifra considerable (1,720 fallecidos), indicando que, aunque los vehículos de cuatro ruedas ofrecen mayor protección, su frecuencia de siniestros graves sigue siendo alta. En contraste, medios de transporte como bicicletas, cuatrimotos o taxis presentan valores mucho menores.

In [0]:
## Ejercicio 13
h_franjas = hechos.withColumn(
    "franja_horaria",
    F.when((F.col("hora")>=6) & (F.col("hora")<12), "Manana")
     .when((F.col("hora")>=12) & (F.col("hora")<18), "Tarde")
     .when((F.col("hora")>=18) & (F.col("hora")<24), "Noche")
     .otherwise("Madrugada")
)
franjas = h_franjas.groupBy("franja_horaria").agg(F.count(F.lit(1)).alias("accidentes")).orderBy("franja_horaria")
display(franjas)

franja_horaria,accidentes
Madrugada,11186
Manana,13580
Noche,26815
Tarde,18854


El análisis de accidentes clasificados por franja horaria muestra que la mayor cantidad de incidentes ocurre durante la noche, con 26,815 accidentes, lo que representa el periodo de mayor riesgo vial. Esta alta incidencia puede asociarse a baja visibilidad, cansancio de los conductores, le sigue la tarde, con 18,854 accidentes, coincidiendo con las horas pico de tráfico y retorno laboral. En la mañana se registran 13,580 accidentes, mientras que la madrugada presenta el menor número (11,186), reflejando la menor movilidad en ese horario.

In [0]:
## Ejercicio 14
fall_dep = victimas.filter(F.lower(F.col("condicion_victima")).like("%fallec%")).groupBy("departamento").agg(F.count(F.lit(1)).alias("fallecidos"))
acc_dep = hechos.groupBy("departamento").agg(F.count(F.lit(1)).alias("accidentes"))
ratio_dep = acc_dep.join(fall_dep, on="departamento", how="left").fillna({"fallecidos":0}).withColumn("ratio_fall_acc", F.col("fallecidos")/F.col("accidentes")).orderBy(F.desc("ratio_fall_acc"))
ratio_dep.write.mode("overwrite").parquet(f"{OUT_BASE}/ratio_fallecidos_por_accidente_departamento")
ratio_dep.show(30, truncate=False)

+-------------+----------+----------+------------------+
|departamento |accidentes|fallecidos|ratio_fall_acc    |
+-------------+----------+----------+------------------+
|21           |96        |221       |2.3020833333333335|
|14           |157       |361       |2.299363057324841 |
|SACATEPÉQUEZ |1556      |3303      |2.122750642673522 |
|QUICHÉ       |674       |1401      |2.078635014836795 |
|QUICHE       |484       |976       |2.0165289256198347|
|8            |69        |137       |1.9855072463768115|
|SOLOLÁ       |921       |1814      |1.9695982627578719|
|16           |315       |607       |1.926984126984127 |
|15           |112       |213       |1.9017857142857142|
|TOTONICAPÁN  |478       |893       |1.8682008368200838|
|22           |182       |333       |1.8296703296703296|
|19           |208       |380       |1.8269230769230769|
|4            |263       |480       |1.8250950570342206|
|PETÉN        |1132      |1994      |1.7614840989399294|
|HUEHUETENANGO|1206      |2082 

El análisis del ratio de fallecidos por accidente por departamento revela diferencias significativas en la gravedad de los siniestros. Los valores más altos se observan en los departamentos identificados con los códigos 21 (2.30) y 14 (2.29), seguidos de Sacatepéquez (2.12) y Quiché (2.08), lo que indica que en promedio, en estos lugares ocurren más de dos fallecidos por accidente, reflejando una alta letalidad vial. En contraste, departamentos con mayor número total de accidentes como Escuintla (1.55) y Jutiapa (1.55) presentan un ratio menor, lo que sugiere que, aunque tienen más incidentes, la proporción de fallecidos por caso es menor.

In [0]:
## Ejercicio 15 - Grupos de edad más afectados

# Crear grupos de edad
v2 = victimas.withColumn("grupo_edad",
    F.when(F.col("edad") < 6, "00-05")
     .when((F.col("edad") >= 6) & (F.col("edad") <= 12), "06-12")
     .when((F.col("edad") >= 13) & (F.col("edad") <= 17), "13-17")
     .when((F.col("edad") >= 18) & (F.col("edad") <= 24), "18-24")
     .when((F.col("edad") >= 25) & (F.col("edad") <= 34), "25-34")
     .when((F.col("edad") >= 35) & (F.col("edad") <= 44), "35-44")
     .when((F.col("edad") >= 45) & (F.col("edad") <= 54), "45-54")
     .when((F.col("edad") >= 55) & (F.col("edad") <= 64), "55-64")
     .otherwise("65+")
)

# Filtrar fallecidos usando fall_les
fall_g = v2.filter(
    (F.col("fall_les") == "Fallecido") | 
    (F.col("fall_les") == "1")
).groupBy("grupo_edad").agg(
    F.count(F.lit(1)).alias("fallecidos")
).orderBy("grupo_edad")

# Filtrar lesionados usando fall_les
les_g = v2.filter(
    (F.col("fall_les") == "Lesionado") | 
    (F.col("fall_les") == "2")
).groupBy("grupo_edad").agg(
    F.count(F.lit(1)).alias("lesionados")
).orderBy("grupo_edad")

# Combinar ambos DataFrames
comp = fall_g.join(les_g, on="grupo_edad", how="outer").fillna(0).orderBy("grupo_edad")

print("=== Comparación de fallecidos y lesionados por grupo de edad ===")
display(comp)


=== Comparación de fallecidos y lesionados por grupo de edad ===


grupo_edad,fallecidos,lesionados
00-05,361,2852
06-12,368,4214
13-17,709,5856
18-24,3306,19695
25-34,4529,20388
35-44,2806,10599
45-54,1598,6060
55-64,1176,3645
65+,3145,9043


El análisis de los grupos de edad más afectados en accidentes muestra que las personas de 25 a 34 años son las más vulnerables, con 4,529 fallecidos y 20,388 lesionados, seguidas por el grupo de 18 a 24 años, que también presenta cifras muy elevadas. Esto refleja que los adultos jóvenes, quienes suelen tener una mayor exposición al tránsito por motivos laborales o de estudio, concentran la mayoría de las víctimas. En contraste, los niños y adolescentes menores de 17 años registran menos fallecidos, aunque un número considerable de lesionados, lo que sugiere una mayor probabilidad de sobrevivir a los accidentes. Finalmente, las personas mayores de 65 años muestran un número elevado de fallecidos (3,145) en relación con los lesionados, evidenciando una mayor fragilidad ante los impactos.

In [0]:
## Ejercicio 16
h_gt = hechos.filter(F.col("municipio")==F.lit("Guatemala"))
v_gt = victimas.filter(F.col("municipio")==F.lit("Guatemala"))
acc_zona = h_gt.groupBy("zona").agg(F.count(F.lit(1)).alias("accidentes"))
fall_zona = v_gt.filter(F.lower(F.col("condicion_victima")).like("%fallec%")).groupBy("zona").agg(F.count(F.lit(1)).alias("fallecidos"))
zonas = acc_zona.join(fall_zona, on="zona", how="outer").fillna(0).orderBy(F.col("zona").cast("int"))
display(zonas)


zona,accidentes,fallecidos
null,0,1942
null,1711,0
1,1643,1825
2,261,299
3,401,466
4,223,218
5,529,564
6,787,829
7,1442,1469
8,278,285


El análisis de los accidentes y fallecidos por zona en el municipio de Guatemala muestra que las zonas 1, 7, 12 y 18 concentran la mayor cantidad de siniestros y muertes, destacando la zona 1 con 1,643 accidentes y 1,825 fallecidos, y la zona 12 con 1,457 accidentes y 1,497 fallecidos. ambién se observa que algunas zonas, como la 6 y la 11, presentan valores altos y consistentes en ambos indicadores, mientras que otras, como la zona 14 y 24, registran cifras significativamente menores. Además, existen registros con valores nulos, que podrían representar casos sin información geográfica completa.

In [0]:
## Ejercicio 17
sexos = vehiculos.select(F.lower(F.col("sexo_conductor")).alias("sexo"))
sexos = sexos.withColumn("sexo_norm",
    F.when(F.col("sexo").like("%masc%") | F.col("sexo").like("%homb%") | (F.col("sexo")=="m"), "Hombre")
     .when(F.col("sexo").like("%fem%") | F.col("sexo").like("%muj%") | (F.col("sexo")=="f"), "Mujer")
     .otherwise("Otro")
)
totales = sexos.groupBy("sexo_norm").agg(F.count(F.lit(1)).alias("n"))
suma = totales.agg(F.sum("n").alias("total")).collect()[0]["total"]
porc = totales.withColumn("porcentaje", (F.col("n")/F.lit(suma))*100)
porc.write.mode("overwrite").parquet(f"{OUT_BASE}/porcentaje_accidentes_por_sexo_conductor")
display(porc)

sexo_norm,n,porcentaje
Otro,20893,19.772867079922396
Hombre,78767,74.54407798230255
Mujer,6005,5.683054937775044


Databricks visualization. Run in Databricks to view.

El análisis de los accidentes según el sexo del conductor revela que la mayoría de los siniestros involucran a conductores hombres, representando aproximadamente el 74.5% del total. En contraste, los conductores mujeres participan en un porcentaje mucho menor, 5.7%, mientras que la categoría “Otro” comprende cerca del 19.8%, posiblemente incluyendo registros no especificados. Estos resultados sugieren que los hombres tienen mayor exposición al riesgo vial, probablemente por una mayor frecuencia de conducción y desplazamientos diarios.